# Urban Heat & Cooling-Priority Mapping — Track A / S2: Heat-Layer Variants

**NUS-ISS Practice Module, Week 2.** One notebook, run top to bottom:

1. **Setup** — install deps, authenticate Earth Engine, mount Drive (all auth up front)
2. **S2.1** — config (AOI, season window)
3. **S2.2** — fetch URA subzones directly from data.gov.sg (no EE asset needed)
4. **S2.3** — subzone ID field check
5. **S2.4** — cloud masking helpers
6. **S2.5** — season-controlled composites (Landsat LST, Sentinel-2 indices) (C4)
7. **S2.6** — composite coverage check
8. **S2.7** — Variant 1: native 30m
9. **S2.8** — Variant 2: bicubic-to-10m baseline
10. **S2.9** — Variant 3: regression-downscaled 10m (C3)
11. **S2.10** — zonal join to URA subzones (CRS checkpoint, silent-drop detection)
12. **S2.11** — verdict (checks dict, PASS/FAIL)
13. **S2.12** — export to Drive
14. **S2.13 / S2.14 (optional)** — poll task, load result CSV

Run cells in order. S2.5 depends on S2.2-S2.4, S2.9 depends on S2.5, S2.10 depends on
S2.7-S2.9. This fills the C3 ablation row for the rank-impact test —
`rank_impact.py` consumes the exported CSV downstream.

---


# SETUP — run once per session

## Setup 1 — Install dependencies

In [1]:
# --- SETUP CELL 1: Install all dependencies used in this notebook ----------
!pip install -q earthengine-api geemap pandas requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.5 MB/s eta 0:00:00


## Setup 2 — Authenticate & initialize Earth Engine

In [2]:
# --- SETUP CELL 2: Authenticate & initialize Earth Engine -------------------
import ee

PROJECT_ID = "nus-iss-urban-heat-sg"  # <-- your GCP project (must match the one
                                      #     used in the Week-1 gates notebook)

if PROJECT_ID == "your-gcp-project-id":
    raise ValueError(
        "PROJECT_ID is still the placeholder. Set it to your actual GCP project ID "
        "(lowercase-with-hyphens, from console.cloud.google.com), then re-run this cell."
    )

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("EE initialized OK, project:", PROJECT_ID)


EE initialized OK, project: nus-iss-urban-heat-sg


## Setup 3 — Mount Google Drive

In [3]:
# --- SETUP CELL 3: Mount Google Drive (needed to read the export later) ----
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


Mounted at /content/drive
Drive mounted at /content/drive


## Setup 4 — Initialize shared results tracker

In [4]:
# --- SETUP CELL 4: Initialize shared results tracker ------------------------
# S2.11's verdict cell writes into this dict, same pattern as Week-1's
# gate_results, so this notebook's outcome can be compiled/compared the same way.
track_a_results = {}
print("track_a_results initialized — populated by the S2.11 verdict cell.")


track_a_results initialized — populated by the S2.11 verdict cell.


---
# S2 — Three heat-layer variants + zonal LST per URA subzone

Native 30m -> bicubic-to-10m baseline -> regression-downscaled 10m (C3), each
reduced to subzone-level mean LST and joined into one output table.


## S2.1 — Config (AOI, season window)

In [5]:
# --- S2 CELL 1: Config -------------------------------------------------------
# NOTE: this assumes SETUP CELL 2 (EE auth) has already run in this session.

# Singapore bounding box — same AOI convention as the Week-1 gates notebook.
sg_bbox = ee.Geometry.Rectangle([103.55, 1.15, 104.10, 1.48])

# Season-controlled, multi-year window (C4). Keep this identical across all
# three variants, and identical to adaptive_capacity_pillar.ipynb.
YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
DRY_SEASON_MONTHS = [4, 5, 10, 11]     # Both inter-monsoon periods (peak-heat conditions:
                                        # weak winds, low cloud, high insolation) — confirmed
                                        # via season_window_diagnostic.ipynb: 45 usable Landsat
                                        # scenes at cloud<70, the strongest of all candidates
                                        # tested. Locked C4 decision.

CLOUD_COVER_MAX = 70                    # Matches Week-1's actual locked gate value (previously
                                         # had this at 20, an unconfirmed guess and the dominant
                                         # cause of an earlier 2-scene composite bug)
S2_CLOUD_PROB_MAX = 40                  # Sentinel-2 s2cloudless threshold

NATIVE_SCALE = 30
TARGET_SCALE = 10
S2_UTM_CRS = "EPSG:32648"               # UTM Zone 48N — covers Singapore, matches native
                                         # Sentinel-2 tiling. Used to give the Landsat composite
                                         # a real projection (see S2 CELL 5) since .median()
                                         # composites don't carry one by default.

REG_SAMPLE_N = 4000                     # points for the OLS fit (Variant 3)
REG_SAMPLE_SEED = 42                    # pinned seed, per eval rules

EXPORT_DESCRIPTION = "heat_layer_variants_subzone"
EXPORT_FOLDER = "urban_heat_sg"
EXPORT_FILE_PREFIX = "heat_variants_subzone"

print(f"AOI: Singapore bbox | Season window: months {DRY_SEASON_MONTHS} across years {YEARS}")


AOI: Singapore bbox | Season window: months [4, 5, 10, 11] across years [2021, 2022, 2023, 2024, 2025, 2026]


## S2.2 — Fetch URA subzones directly from data.gov.sg

Same source and same fetch pattern as Week-1 G3.8 (`SUBZONE_DATASET_ID =
"d_8594ae9ff96d0c708bc2af633048edfb"`) — no pre-uploaded EE asset required.
Downloads the GeoJSON to local Colab disk, loads it locally, sanitizes any
property names containing `.` (the data.gov.sg export includes ArcGIS-style
fields like `SHAPE.AREA`/`SHAPE.LEN`, which Earth Engine rejects outright as
invalid property names), then uploads it via `ee.FeatureCollection()`, which
accepts a raw GeoJSON dict natively — no geemap conversion helper needed.
This re-uploads the ~330-feature file to EE every run instead of persisting
it as a named EE asset — if you'd rather manage a persistent asset, ingest it
once via the Code Editor Assets tab and swap the last line of this cell for
`subzones = ee.FeatureCollection("projects/.../assets/...")`.


In [6]:
# --- S2 CELL 2: Fetch URA subzones from data.gov.sg (no EE asset needed) ---
import requests
import json

SUBZONE_DATASET_ID = "d_8594ae9ff96d0c708bc2af633048edfb"  # MP19 Subzone Boundary (No Sea), GEOJSON
SUBZONE_LOCAL_PATH = "/content/ura_subzones.geojson"

def fetch_datagovsg_geojson(dataset_id, out_path):
    poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download"
    r = requests.get(poll_url)
    r.raise_for_status()
    payload = r.json()
    if payload.get("code") != 0:
        raise RuntimeError(f"data.gov.sg API error: {payload.get('errMsg')}")
    download_url = payload["data"]["url"]
    geojson_bytes = requests.get(download_url).content
    with open(out_path, "wb") as f:
        f.write(geojson_bytes)
    return out_path

try:
    subzone_path = fetch_datagovsg_geojson(SUBZONE_DATASET_ID, SUBZONE_LOCAL_PATH)
    print(f"Downloaded subzones GeoJSON -> {subzone_path}")
except Exception as e:
    print(f"⚠️  Auto-download failed ({e}). Manual fallback:")
    print("   1. Download GeoJSON from https://data.gov.sg/datasets/d_8594ae9ff96d0c708bc2af633048edfb/view")
    print(f"   2. Upload it to {SUBZONE_LOCAL_PATH} in the Colab file browser (left sidebar)")
    print(f"   3. Re-run this cell — it will find the file and skip the failed download.")
    raise

with open(subzone_path) as f:
    subzone_geojson = json.load(f)

n_features_local = len(subzone_geojson.get("features", []))
print(f"Parsed GeoJSON locally: {n_features_local} features (before upload to EE)")
if n_features_local == 0:
    raise ValueError("Downloaded GeoJSON has zero features — check the file/download before proceeding.")

# EE rejects property names containing '.' (e.g. ArcGIS-style SHAPE.AREA / SHAPE.LEN
# fields, common in URA/data.gov.sg exports) with "Invalid property name". Sanitize
# locally before upload rather than letting the upload fail deep in ee.data internals.
renamed_props = set()
for feature in subzone_geojson.get("features", []):
    props = feature.get("properties", {})
    for old_key in list(props.keys()):
        if "." in old_key:
            new_key = old_key.replace(".", "_")
            props[new_key] = props.pop(old_key)
            renamed_props.add(f"{old_key} -> {new_key}")

if renamed_props:
    print(f"⚠️  Sanitized {len(renamed_props)} property name(s) containing '.': {sorted(renamed_props)}")
else:
    print("No dotted property names found — nothing to sanitize.")

subzones = ee.FeatureCollection(subzone_geojson)
n_subzones_total = subzones.size().getInfo()
print(f"Loaded {n_subzones_total} subzones into an ee.FeatureCollection")

if n_subzones_total != n_features_local:
    print(f"⚠️  EE feature count ({n_subzones_total}) != local parsed count ({n_features_local}) — "
          f"investigate before trusting downstream joins.")

sample_props = subzones.first().propertyNames().getInfo()
print("Sample subzone property names:", sample_props)


Downloaded subzones GeoJSON -> /content/ura_subzones.geojson
Parsed GeoJSON locally: 332 features (before upload to EE)
⚠️  Sanitized 2 property name(s) containing '.': ['SHAPE.AREA -> SHAPE_AREA', 'SHAPE.LEN -> SHAPE_LEN']
Loaded 332 subzones into an ee.FeatureCollection
Sample subzone property names: ['system:index', 'PLN_AREA_C', 'SHAPE_LEN', 'REGION_N', 'CA_IND', 'INC_CRC', 'SHAPE_AREA', 'OBJECTID', 'SUBZONE_C', 'SUBZONE_NO', 'PLN_AREA_N', 'REGION_C', 'FMEL_UPD_D', 'SUBZONE_N']


## S2.3 — Subzone ID field check

Same "never assume, always check" checkpoint the Week-1 notebook uses for
CRS (G3.8) — here for the subzone name/ID field, since a wrong field silently
breaks the join in S2.10 rather than raising an error.


In [7]:
# --- S2 CELL 3: Subzone ID field check --------------------------------------
SUBZONE_ID_PROPERTY = "SUBZONE_N"   # <-- set this to match the GeoJSON's actual ID field

if SUBZONE_ID_PROPERTY not in sample_props:
    candidates = [p for p in sample_props if "name" in p.lower() or "subzone" in p.lower() or "sz" in p.lower()]
    print(f"⚠️  '{SUBZONE_ID_PROPERTY}' not found in subzone properties.")
    print(f"   Candidates from the file: {candidates if candidates else sample_props}")
    raise ValueError("Fix SUBZONE_ID_PROPERTY above to match the GeoJSON, then re-run.")
else:
    print(f"✅ SUBZONE_ID_PROPERTY = '{SUBZONE_ID_PROPERTY}' confirmed present.")


✅ SUBZONE_ID_PROPERTY = 'SUBZONE_N' confirmed present.


## S2.4 — Cloud masking helpers

In [8]:
# --- S2 CELL 4: Cloud masking helpers ---------------------------------------
def mask_landsat_c2l2(image):
    """Cloud/shadow/snow mask using QA_PIXEL bits, per USGS C2 L2 spec."""
    qa = image.select("QA_PIXEL")
    dilated_cloud = 1 << 1
    cirrus = 1 << 2
    cloud = 1 << 3
    shadow = 1 << 4
    snow = 1 << 5
    mask = (
        qa.bitwiseAnd(dilated_cloud).eq(0)
        .And(qa.bitwiseAnd(cirrus).eq(0))
        .And(qa.bitwiseAnd(cloud).eq(0))
        .And(qa.bitwiseAnd(shadow).eq(0))
        .And(qa.bitwiseAnd(snow).eq(0))
    )
    sat_mask = image.select("QA_RADSAT").eq(0)
    return image.updateMask(mask).updateMask(sat_mask)


def scale_st_b10_celsius(image):
    """Apply Collection 2 Level 2 scale factors, convert Kelvin -> Celsius."""
    lst_c = (
        image.select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("LST_C")
    )
    return image.addBands(lst_c, overwrite=True)


def mask_s2_clouds(cloud_prob_image):
    """s2cloudless probability mask for Sentinel-2 L2A."""
    return cloud_prob_image.select("probability").lt(S2_CLOUD_PROB_MAX)


def date_filter_for_years_months(collection, years, months):
    """Union filter: keep images that fall in ANY (year, month) combo. Used
    to build season-controlled, multi-year composites (C4) — a plain
    filterDate(start, end) would mix wet/dry-season thermal regimes."""
    filters = []
    for y in years:
        for m in months:
            start = ee.Date.fromYMD(y, m, 1)
            end = start.advance(1, "month")
            filters.append(ee.Filter.date(start, end))
    return collection.filter(ee.Filter.Or(*filters))


print("Cloud masking + season-filter helpers defined.")


Cloud masking + season-filter helpers defined.


## S2.5 — Season-controlled composites (Landsat LST, Sentinel-2 indices)

In [9]:
# --- S2 CELL 5: Landsat 8/9 LST composite (season-controlled, C4) ----------
l8 = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(sg_bbox)
    .filter(ee.Filter.lt("CLOUD_COVER", CLOUD_COVER_MAX))
)
l9 = (
    ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
    .filterBounds(sg_bbox)
    .filter(ee.Filter.lt("CLOUD_COVER", CLOUD_COVER_MAX))
)

l89_all = l8.merge(l9)
print("Landsat 8+9 scenes (pre season-filter, pre-mask):", l89_all.size().getInfo())

l89 = date_filter_for_years_months(l89_all, YEARS, DRY_SEASON_MONTHS)
print(f"Landsat 8+9 scenes after season filter (months={DRY_SEASON_MONTHS}, years={YEARS}):", l89.size().getInfo())

l89 = l89.map(mask_landsat_c2l2).map(scale_st_b10_celsius)
lst_30m_raw = l89.select("LST_C").median().rename("LST_C").clip(sg_bbox)

# A .median() composite over an ImageCollection does not carry a usable default
# projection (confirmed earlier: printing lst_30m.projection() returned plain
# EPSG:4326, not a real meters-based grid) — every downstream step that relied
# on "lst_30m.projection()" as if it were a proper 30m UTM grid was silently
# using this degenerate default instead. Fixed at the root here, once, so
# every variant below inherits a real projection rather than needing a patch
# each place lst_30m gets used.
lst_30m = lst_30m_raw.reproject(crs=S2_UTM_CRS, scale=NATIVE_SCALE)

print("Landsat merged usable scenes (post-mask):", l89.size().getInfo())


Landsat 8+9 scenes (pre season-filter, pre-mask): 254
Landsat 8+9 scenes after season filter (months=[4, 5, 10, 11], years=[2021, 2022, 2023, 2024, 2025, 2026]): 45
Landsat merged usable scenes (post-mask): 45


In [10]:
# --- S2 CELL 6: Sentinel-2 index composite (season-controlled, C4) ---------
s2_sr = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(sg_bbox)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
)
s2_sr = date_filter_for_years_months(s2_sr, YEARS, DRY_SEASON_MONTHS)
print("Sentinel-2 scenes after season filter (pre-mask):", s2_sr.size().getInfo())

s2_cloud_prob = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(sg_bbox)
s2_cloud_prob = date_filter_for_years_months(s2_cloud_prob, YEARS, DRY_SEASON_MONTHS)

joined = ee.Join.saveFirst("cloud_mask").apply(
    primary=s2_sr,
    secondary=s2_cloud_prob,
    condition=ee.Filter.equals(leftField="system:index", rightField="system:index"),
)

def _mask_and_index(img):
    img = ee.Image(img)
    cloud_img = ee.Image(img.get("cloud_mask"))
    clear_mask = mask_s2_clouds(cloud_img)
    img = img.updateMask(clear_mask)
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndbi = img.normalizedDifference(["B11", "B8"]).rename("NDBI")
    ndwi = img.normalizedDifference(["B3", "B8"]).rename("NDWI")
    return img.addBands([ndvi, ndbi, ndwi])

s2_indexed = ee.ImageCollection(joined).map(_mask_and_index)
s2_indices_10m = s2_indexed.select(["NDVI", "NDBI", "NDWI"]).median().clip(sg_bbox)

print("Sentinel-2 usable scenes (post-mask):", s2_indexed.size().getInfo())


Sentinel-2 scenes after season filter (pre-mask): 59
Sentinel-2 usable scenes (post-mask): 59


## S2.6 — Composite coverage check

Same purpose as the Week-1 G1.4/G1.6 coverage + verdict pair: confirm the
composite actually has usable pixel coverage over the AOI before spending
compute on the regression variant downstream.


In [11]:
# --- S2 CELL 7: Composite coverage check -------------------------------------
def coverage_fraction(image, band_name, aoi, scale):
    mask_img = image.select(band_name).mask()
    stats = mask_img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=scale,
        maxPixels=1e10,
        bestEffort=True,
    )
    return ee.Number(stats.get(band_name))

lst_coverage = coverage_fraction(lst_30m, "LST_C", sg_bbox, NATIVE_SCALE).getInfo()
ndvi_coverage = coverage_fraction(s2_indices_10m, "NDVI", sg_bbox, TARGET_SCALE).getInfo()

print(f"Landsat LST_C valid-pixel coverage over AOI: {lst_coverage*100:.1f}%")
print(f"Sentinel-2 NDVI valid-pixel coverage over AOI: {ndvi_coverage*100:.1f}%")

COVERAGE_THRESHOLD = 0.90
coverage_ok = lst_coverage >= COVERAGE_THRESHOLD and ndvi_coverage >= COVERAGE_THRESHOLD

if coverage_ok:
    print(f"\n✅ Coverage OK (>= {COVERAGE_THRESHOLD*100:.0f}%). Proceeding to variants.")
else:
    print(f"\n⚠️  Coverage below {COVERAGE_THRESHOLD*100:.0f}% threshold:")
    print("   - Widen DRY_SEASON_MONTHS or YEARS in S2 CELL 1")
    print("   - Raise CLOUD_COVER_MAX / S2_CLOUD_PROB_MAX pre-filters")
    print("   - Confirm the season window isn't unusually cloudy this range")


Landsat LST_C valid-pixel coverage over AOI: 94.3%
Sentinel-2 NDVI valid-pixel coverage over AOI: 100.0%

✅ Coverage OK (>= 90%). Proceeding to variants.


## S2.7 — Variant 1: native 30m

In [12]:
# --- S2 CELL 8: Variant 1 — native 30m ---------------------------------------
lst_native30 = lst_30m.rename("lst_native30")
print("Variant 1 (native 30m) defined.")


Variant 1 (native 30m) defined.


## S2.8 — Variant 2: bicubic-to-10m baseline

In [13]:
# --- S2 CELL 9: Variant 2 — bicubic-to-10m baseline --------------------------
# Pure interpolation, no new information — this is the "dumb" baseline the
# regression variant (S2.9, next section) needs to beat on both RMSE and rank impact.
#
# lst_30m already carries a real UTM projection (fixed at the source in S2 CELL 5),
# so bicubic resampling here reads true 30m pixels rather than a degenerate default
# grid — this is what fixed a bug where bicubic10 barely varied at all across
# native30's full range (confirmed via the diagnostic notebook).
lst_bicubic10 = (
    lst_30m.resample("bicubic")
    .reproject(crs=S2_UTM_CRS, scale=TARGET_SCALE)
    .rename("lst_bicubic10")
)
print("Variant 2 (bicubic 10m) defined.")


Variant 2 (bicubic 10m) defined.


## S2.9 — Variant 3: regression-downscaled 10m (C3)

TsHARP-style downscaling, matching the approach validated in Week-1 G4:

1. Aggregate S2 indices to 30m (mean) to match Landsat resolution.
2. Fit `LST_30m ~ NDVI_30m + NDBI_30m + NDWI_30m` by OLS (with intercept).
3. Apply coefficients to native 10m S2 indices for the raw 10m prediction.
4. Residual correction: `residual_30m = lst_30m - predicted_at_30m`, resample
   to 10m (bilinear), add back so the output stays consistent with the coarse
   observation (mass conservation) — same caveat as Week-1 G4.5: this R² is
   measured at 30m, since no independent 10m LST ground truth exists.


In [14]:
# --- S2 CELL 10: Variant 3 — regression-downscaled 10m (C3) ------------------
# reduceResolution requires the INPUT to already have a concrete, fine-resolution
# default projection — a median() composite doesn't carry one (that's what "does
# not have a valid default projection" means), so we set one explicitly at native
# 10m in a real metric CRS BEFORE calling reduceResolution, then reproject onto
# the coarser 30m Landsat grid to actually perform the mean aggregation. Doing
# reproject-to-30m before reduceResolution (as an earlier version of this cell
# did) is backwards — it throws away the fine pixels reduceResolution needs.
s2_indices_10m_fine = s2_indices_10m.reproject(crs=S2_UTM_CRS, scale=TARGET_SCALE)

s2_indices_30m = (
    s2_indices_10m_fine
    .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=64)
    .reproject(crs=S2_UTM_CRS, scale=NATIVE_SCALE)
)

training_stack = lst_30m.addBands(s2_indices_30m).addBands(ee.Image.constant(1).rename("CONST"))
training_bands = ["CONST", "NDVI", "NDBI", "NDWI", "LST_C"]

samples = training_stack.select(training_bands).sample(
    region=sg_bbox,
    scale=NATIVE_SCALE,
    numPixels=REG_SAMPLE_N,
    seed=REG_SAMPLE_SEED,
    geometries=False,
    tileScale=4,
)
n_samples = samples.size().getInfo()
print(f"Regression training samples drawn: {n_samples} (requested {REG_SAMPLE_N})")

if n_samples < 30:
    print("⚠️  Very few valid samples — regression coefficients below will be noisy.")
    print("   Check AOI/season-filter coverage in S2.6 before trusting Variant 3.")

regression = samples.reduceColumns(
    reducer=ee.Reducer.linearRegression(numX=4, numY=1),
    selectors=training_bands,
)
coeffs = ee.Array(regression.get("coefficients")).project([0])
b0 = ee.Number(coeffs.get([0]))
b1 = ee.Number(coeffs.get([1]))
b2 = ee.Number(coeffs.get([2]))
b3 = ee.Number(coeffs.get([3]))

print(f"Fitted coefficients — intercept={b0.getInfo():.3f}, "
      f"NDVI={b1.getInfo():.3f}, NDBI={b2.getInfo():.3f}, NDWI={b3.getInfo():.3f}")

predicted_10m = (
    s2_indices_10m.select("NDVI").multiply(b1)
    .add(s2_indices_10m.select("NDBI").multiply(b2))
    .add(s2_indices_10m.select("NDWI").multiply(b3))
    .add(b0)
    .rename("LST_pred_10m")
)

predicted_30m = (
    s2_indices_30m.select("NDVI").multiply(b1)
    .add(s2_indices_30m.select("NDBI").multiply(b2))
    .add(s2_indices_30m.select("NDWI").multiply(b3))
    .add(b0)
    .rename("LST_pred_30m")
)

residual_30m = lst_30m.subtract(predicted_30m).rename("residual_30m")
residual_10m = residual_30m.resample("bilinear").reproject(crs=S2_UTM_CRS, scale=TARGET_SCALE)

lst_regress10 = predicted_10m.add(residual_10m).rename("lst_regress10")
print("Variant 3 (regression-downscaled 10m) defined.")


Regression training samples drawn: 3777 (requested 4000)
Fitted coefficients — intercept=37.493, NDVI=-24.745, NDBI=14.537, NDWI=-31.806
Variant 3 (regression-downscaled 10m) defined.


## S2.10 — Zonal join to URA subzones

CRS checkpoint + silent-drop detection, same discipline as Week-1 G3.8/G3.9 —
`reduceRegions` returns a feature with a null `mean` (not an error) when a
subzone has no valid pixels in-mask, so those need to be counted explicitly,
not assumed away. Pulls the joined table down with a single `.getInfo()`
and does all counting locally in Python — Earth Engine doesn't cache
across separate calls, so counting nulls with repeated server-side
`.filter().size().getInfo()` calls (as an earlier version of this cell did)
re-runs the entire lazy computation graph, including the expensive
`lst_regress10` chain, once per call. If this cell feels slow, that
redundant recomputation is very likely why.


In [15]:
# --- S2 CELL 11: Zonal join (CRS checkpoint + silent-drop detection) -------
lst_native30_proj = lst_native30.projection().getInfo()
lst_regress10_proj = lst_regress10.projection().getInfo()
print(f"lst_native30 projection: {lst_native30_proj['crs']}")
print(f"lst_regress10 projection: {lst_regress10_proj['crs']}")
print("(reduceRegions reprojects subzone geometries to each image's CRS internally —")
print(" both bands above should be a Landsat/UTM-derived CRS; if they don't match,")
print(" stop and check the .reproject() calls in S2.8/S2.9 before trusting the join.)")


def zonal_mean(image, band_name, scale):
    reduced = image.rename(band_name).reduceRegions(
        collection=subzones,
        reducer=ee.Reducer.mean(),
        scale=scale,
        tileScale=4,
    )
    return reduced.map(lambda f: f.set(band_name, f.get("mean")))


z_native = zonal_mean(lst_native30, "lst_native30", NATIVE_SCALE)
z_bicubic = zonal_mean(lst_bicubic10, "lst_bicubic10", TARGET_SCALE)
z_regress = zonal_mean(lst_regress10, "lst_regress10", TARGET_SCALE)

join_filter = ee.Filter.equals(leftField=SUBZONE_ID_PROPERTY, rightField=SUBZONE_ID_PROPERTY)

merged1 = ee.FeatureCollection(
    ee.Join.inner().apply(z_native, z_bicubic, join_filter).map(
        lambda pair: ee.Feature(pair.get("primary")).set(
            "lst_bicubic10", ee.Feature(pair.get("secondary")).get("lst_bicubic10")
        )
    )
)
merged2 = ee.FeatureCollection(
    ee.Join.inner().apply(merged1, z_regress, join_filter).map(
        lambda pair: ee.Feature(pair.get("primary")).set(
            "lst_regress10", ee.Feature(pair.get("secondary")).get("lst_regress10")
        )
    )
)

final_table = merged2.map(
    lambda f: ee.Feature(None, {
        "subzone_id": f.get(SUBZONE_ID_PROPERTY),
        "lst_native30": f.get("lst_native30"),
        "lst_bicubic10": f.get("lst_bicubic10"),
        "lst_regress10": f.get("lst_regress10"),
    })
)

# Pull the joined table down ONCE and do all counting locally in Python. EE does
# not cache across separate .getInfo()/.size() calls — calling those 4 separate
# times (as an earlier version of this cell did: n_joined, then 3 null-filter
# counts) re-triggers the ENTIRE lazy computation graph from scratch each time,
# including the expensive lst_regress10 chain (composite -> reduceResolution ->
# regression -> residual correction) evaluated at 10m over all of Singapore.
# One .getInfo() here instead of four cuts that redundant recomputation out —
# this is very likely why S2.10 felt slow.
final_records = final_table.getInfo()["features"]
final_rows = [f["properties"] for f in final_records]

n_joined = len(final_rows)
print(f"\nJoin summary: {n_subzones_total} subzones in asset -> {n_joined} retained after inner join")

null_native = sum(1 for r in final_rows if r.get("lst_native30") is None)
null_bicubic = sum(1 for r in final_rows if r.get("lst_bicubic10") is None)
null_regress = sum(1 for r in final_rows if r.get("lst_regress10") is None)

print(f"Rows with null lst_native30: {null_native}")
print(f"Rows with null lst_bicubic10: {null_bicubic}")
print(f"Rows with null lst_regress10: {null_regress}")
if null_native or null_bicubic or null_regress:
    print("⚠️  Non-zero nulls are EXPECTED for tiny/sliver subzones at 10m — not a bug on its own,")
    print("   but confirm this count is small relative to total subzones, not most of them.")


lst_native30 projection: EPSG:32648
lst_regress10 projection: EPSG:32648
(reduceRegions reprojects subzone geometries to each image's CRS internally —
 both bands above should be a Landsat/UTM-derived CRS; if they don't match,
 stop and check the .reproject() calls in S2.8/S2.9 before trusting the join.)

Join summary: 332 subzones in asset -> 332 retained after inner join
Rows with null lst_native30: 0
Rows with null lst_bicubic10: 0
Rows with null lst_regress10: 0


## S2.11 — Verdict

In [16]:
# --- S2 CELL 12: Verdict ------------------------------------------------------
print("\n--- S2 Verdict ---")

s2_checks = {
    "Composite coverage >= 90% (LST and NDVI)": coverage_ok,
    "Regression sample count sufficient (n>=30)": n_samples >= 30,
    "Zonal join retained subzones (non-zero)": n_joined > 0,
    "No majority silent-drop on any variant column": max(null_native, null_bicubic, null_regress) < n_subzones_total * 0.5,
    "All three variant columns present": n_joined > 0 and set(["subzone_id", "lst_native30", "lst_bicubic10", "lst_regress10"]).issubset(final_rows[0].keys()),
}

for check, passed in s2_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

s2_pass = all(s2_checks.values())
if s2_pass:
    print("\n✅ S2 PASS: three heat-layer variants built and joined to subzones. Proceed to export.")
else:
    print("\n⚠️  S2 FAIL: fix flagged step(s) above before exporting / feeding rank_impact.py.")

track_a_results["S2_heat_variants"] = {
    "status": "PASS" if s2_pass else "FAIL",
    "checks": s2_checks,
    "n_subzones_total": n_subzones_total,
    "n_subzones_joined": n_joined,
    "n_null_native30": null_native,
    "n_null_bicubic10": null_bicubic,
    "n_null_regress10": null_regress,
}



--- S2 Verdict ---
  [PASS] Composite coverage >= 90% (LST and NDVI)
  [PASS] Regression sample count sufficient (n>=30)
  [PASS] Zonal join retained subzones (non-zero)
  [PASS] No majority silent-drop on any variant column
  [PASS] All three variant columns present

✅ S2 PASS: three heat-layer variants built and joined to subzones. Proceed to export.


## S2.12 — Export to Google Drive

In [17]:
# --- S2 CELL 13: Export to Drive ---------------------------------------------
task = ee.batch.Export.table.toDrive(
    collection=final_table,
    description=EXPORT_DESCRIPTION,
    folder=EXPORT_FOLDER,
    fileNamePrefix=EXPORT_FILE_PREFIX,
    fileFormat="CSV",
)
task.start()
print(f"Export task started: {task.id}")
print("Monitor at https://code.earthengine.google.com/tasks")
print("Output columns: subzone_id, lst_native30, lst_bicubic10, lst_regress10")


Export task started: ERWET4B3LQK6KSECBUFXTIY2
Monitor at https://code.earthengine.google.com/tasks
Output columns: subzone_id, lst_native30, lst_bicubic10, lst_regress10


## S2.13 (optional) — Poll task status from this notebook

In [18]:
# --- S2 CELL 14 (optional): Poll export task status --------------------------
import time

while True:
    status = task.status()
    state = status.get("state")
    print(state)
    if state in ("COMPLETED", "FAILED", "CANCELLED"):
        print(status)
        break
    time.sleep(15)


READY
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
COMPLETED
{'state': 'COMPLETED', 'description': 'heat_layer_variants_subzone', 'priority': 100, 'creation_timestamp_ms': 1784524368686, 'update_timestamp_ms': 1784524472780, 'start_timestamp_ms': 1784524373127, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/13liO2Ds_78LrYOVv0nfj7DvlOj50x4rK'], 'attempt': 1, 'batch_eecu_usage_seconds': 8593.8291015625, 'id': 'ERWET4B3LQK6KSECBUFXTIY2', 'name': 'projects/nus-iss-urban-heat-sg/operations/ERWET4B3LQK6KSECBUFXTIY2'}


## S2.14 (optional) — Load the exported CSV from mounted Drive

In [19]:
# --- S2 CELL 15 (optional): Load result CSV -----------------------------------
import pandas as pd

csv_path = f"/content/drive/MyDrive/{EXPORT_FOLDER}/{EXPORT_FILE_PREFIX}.csv"
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows from {csv_path}")
df.head()


Loaded 332 rows from /content/drive/MyDrive/urban_heat_sg/heat_variants_subzone.csv


,system:index,lst_bicubic10,lst_native30,lst_regress10,subzone_id,.geo
0,0_0_0,39.363552,39.361782,39.385393,DEPOT ROAD,"{""type"":""MultiPoint"",""coordinates"":[]}"
1,1_1_1,41.839853,41.841607,41.837878,BUKIT MERAH,"{""type"":""MultiPoint"",""coordinates"":[]}"
2,2_2_2,42.508228,42.505996,42.558938,CHINATOWN,"{""type"":""MultiPoint"",""coordinates"":[]}"
3,3_3_3,39.375655,39.387851,39.324798,PHILLIP,"{""type"":""MultiPoint"",""coordinates"":[]}"
4,4_4_4,38.665122,38.665300,38.655310,RAFFLES PLACE,"{""type"":""MultiPoint"",""coordinates"":[]}"
